# Atividade final - Análise de corpora literários

Neste projeto final, você realizará uma pequena análise de dados linguísticos do
início ao fim.

Trabalharemos com três conjuntos de textos literários do
[Brown Corpus](https://www.nltk.org/howto/corpus.html). Ao longo da atividade, você utilizará ferramentas
vistas durante o curso para:

- conhecer e explorar os dados com [pandas](https://pandas.pydata.org/docs/);
- visualizar características dos corpora com [Seaborn](https://seaborn.pydata.org/);
- extrair informações linguísticas com [spaCy](https://spacy.io/usage);
- comparar grupos por meio de um teste de permutação com [SciPy](https://docs.scipy.org/doc/scipy/reference/stats.html);
- realizar uma análise de variância com [statsmodels](https://www.statsmodels.org/stable/index.html).

Boa parte das decisões de análise ficará por sua conta. As instruções indicarão
o problema a ser resolvido, mas nem sempre qual método ou função de Python deve
ser utilizado. Sempre que uma ferramenta específica for sugerida, você encontrará
um link para a documentação oficial correspondente.


## 1. Fonte e carregamento dos dados

Os três conjuntos de textos vêm do [Brown Corpus](https://www.nltk.org/howto/corpus.html), distribuído
pelo [NLTK](https://www.nltk.org/). O código abaixo carrega as categorias
`adventure`, `mystery` e `romance` e as transforma em DataFrames do
[pandas](https://pandas.pydata.org/docs/).

Esta parte do código é fornecida e não precisa ser modificada.


In [ ]:
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
from nltk.corpus import brown


In [ ]:
nltk.download("brown")


def load_brown_category(category):
    records = []

    for fileid in brown.fileids(categories=category):
        text = " ".join(brown.words(fileids=fileid))

        records.append(
            {
                "fileid": fileid,
                "genre": category,
                "text": text,
            }
        )

    return pd.DataFrame(records)


adventure = load_brown_category("adventure")
mystery = load_brown_category("mystery")
romance = load_brown_category("romance")


## 2. Primeiros passos com os dados

Os objetos `adventure`, `mystery` e `romance` são
[`DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) do pandas contendo os três corpora que utilizaremos.
Os textos vêm do [Brown Corpus do NLTK](https://www.nltk.org/howto/corpus.html).

Antes de realizar qualquer análise linguística ou estatística, conheça os dados.

Explore os três DataFrames e investigue, pelo menos:

- sua estrutura;
- o número de observações;
- as variáveis disponíveis;
- os tipos das variáveis;
- a presença de valores ausentes;
- alguns exemplos dos textos.

Compare também o tamanho dos três corpora.


## 3. Unificando e verificando o corpus

Para facilitar as próximas etapas, reúna os três DataFrames em uma única tabela chamada `corpus`, contendo todos os textos de `adventure`, `mystery` e `romance`. 

Se precisar revisar como empilhar DataFrames, consulte [`pandas.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html). **Atenção:** certifique-se de redefinir o índice ao concatenar as tabelas (utilizando o argumento `ignore_index=True`) para evitar índices duplicados nas etapas de visualização e modelagem.

Em seguida, verifique a integridade do DataFrame unificado:

- quantos textos existem em cada gênero;
- se o índice é único e contínuo;
- se existem valores ausentes ou textos duplicados.

### 3.1. Uma primeira medida do tamanho dos textos

Antes de realizar qualquer processamento linguístico, podemos obter algumas
medidas simples diretamente da coluna `text`.

Crie duas novas variáveis:

- `n_characters`: número de caracteres do texto;
- `n_tokens_simple`: número de elementos obtidos ao separar o texto por espaços.

Neste momento, `n_tokens_simple` é apenas uma medida aproximada do tamanho do
texto. Não estamos afirmando que cada elemento separado por espaço corresponde
necessariamente a uma palavra linguisticamente bem definida.

Para revisar operações com texto no pandas, consulte a documentação sobre
[working with text data](https://pandas.pydata.org/docs/user_guide/text.html). Depois, explore essas duas variáveis
utilizando estatísticas descritivas; [`DataFrame.describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
pode ser uma referência útil.


### 3.2. Comparando os três corpora

Compare `adventure`, `mystery` e `romance` em relação ao tamanho dos textos.

Produza pelo menos uma visualização que permita observar a distribuição de uma
das medidas criadas anteriormente nos três gêneros. A documentação do Seaborn
sobre [visualização de distribuições](https://seaborn.pydata.org/tutorial/distributions.html) pode ajudar na escolha do
gráfico.

Em seguida, responda:

1. Os três corpora possuem textos de tamanhos aproximadamente semelhantes?
2. Existe algum texto que se destaque dos demais?
3. A média parece representar bem cada distribuição?
4. Há alguma característica dos dados que você gostaria de investigar antes de
   realizar uma comparação estatística?


## 4. Extração de características linguísticas

Agora utilizaremos o [spaCy](https://spacy.io/usage) para obter informações
linguísticas dos textos.

O código abaixo carrega o modelo de inglês que será utilizado durante a análise.
Se quiser entender melhor como os pipelines pré-treinados funcionam, consulte a
documentação de [modelos e pipelines do spaCy](https://spacy.io/usage/models).


In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

### 4.1. Processando os textos

Use o modelo `nlp` para processar os textos da coluna `text`.

Antes de aplicar o procedimento a todo o corpus, escolha um único texto e explore
o objeto produzido pelo spaCy.

Observe especialmente:

- os tokens;
- as sentenças ([`Doc.sents`](https://spacy.io/api/doc));
- as classes gramaticais ([`Token.pos_`](https://spacy.io/api/token));
- os lemas ([`Token.lemma_`](https://spacy.io/api/token)).

Use essa exploração apenas para se familiarizar com a estrutura produzida pelo
spaCy. As páginas de [`Doc`](https://spacy.io/api/doc) e [`Token`](https://spacy.io/api/token) são as
principais referências para os objetos que você está manipulando.


### 4.2. Variáveis linguísticas

Agora processe todos os textos do DataFrame `corpus` (dica: [`nlp.pipe()`](https://spacy.io/usage/processing-pipelines#processing) processa coleções de textos de forma mais eficiente) e crie as seguintes colunas:

- `n_sentences`: número total de sentenças no texto ([`Doc.sents`](https://spacy.io/api/doc#sents));
- `mean_sentence_length`: razão entre o número total de palavras do texto e `n_sentences`;
- `noun_prop`: proporção de palavras classificadas como substantivos comuns (`Token.pos_ == "NOUN"`);
- `verb_prop`: proporção de palavras classificadas como verbos principais (`Token.pos_ == "VERB"`);
- `adj_prop`: proporção de palavras classificadas como adjetivos (`Token.pos_ == "ADJ"`).

**Critérios operacionais importantes:**
1. Considere como **palavra** apenas tokens alfabéticos, filtrando-os com [`Token.is_alpha`](https://spacy.io/api/token#attributes) (isso exclui sinais de pontuação, números e espaços).
2. Calcule todas as proporções (`noun_prop`, `verb_prop` e `adj_prop`) dividindo a contagem da respectiva classe gramatical pelo **número total de palavras alfabéticas (`is_alpha`)** do texto.

### 4.3. Explore as novas variáveis

Examine as características linguísticas que você acabou de criar.

Compare `adventure`, `mystery` e `romance` utilizando estatísticas descritivas e
visualizações.

Tente responder:

1. Qual dos três gêneros apresenta sentenças mais longas, em média?
2. Há diferenças aparentes nas proporções de substantivos, verbos ou adjetivos?
3. As distribuições apresentam assimetrias ou valores extremos?
4. Qual dessas diferenças parece mais interessante para investigar
   estatisticamente?

Registre suas principais observações antes de realizar qualquer teste estatístico.


### 4.4. Relações entre as características linguísticas

Até aqui, comparamos as características linguísticas principalmente entre os
gêneros. Também podemos investigar como essas características se relacionam
entre si.

Selecione as variáveis numéricas produzidas durante a análise linguística, como
`mean_sentence_length`, `noun_prop`, `verb_prop` e `adj_prop`.

Calcule uma matriz de correlação entre essas variáveis com
[`DataFrame.corr()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) e represente o resultado com um
[`seaborn.heatmap()`](https://seaborn.pydata.org/generated/seaborn.heatmap.html).

Em seguida:

1. identifique a associação positiva mais forte;
2. identifique a associação negativa mais forte;
3. escolha uma dessas relações e examine-a também com um gráfico apropriado para
   duas variáveis numéricas; [`seaborn.scatterplot()`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) pode
   ser usado como referência;

Lembre-se de que uma correlação descreve uma associação entre variáveis e,
isoladamente, não estabelece uma relação causal.


## 5. Teste de permutação

A análise exploratória permitiu observar possíveis diferenças entre os gêneros.
Agora investigaremos uma comparação específica por meio de inferência estatística.

Considere a variável `mean_sentence_length` e concentre-se nos gêneros `mystery`
e `romance`.

Queremos investigar se esses dois grupos apresentam diferenças no comprimento
médio das sentenças.


### 5.1. Hipóteses e diferença observada

Antes de realizar o teste:

1. escreva a hipótese nula $(H_0)$;
2. escreva a hipótese alternativa $(H_1)$;
3. separe os valores de `mean_sentence_length` dos dois gêneros;
4. calcule a diferença observada entre as médias.

Defina a diferença como:

$$
\bar{x}_{mystery} - \bar{x}_{romance}
$$

### 5.2. Testando a diferença com SciPy

Utilize [`scipy.stats.permutation_test()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html) para realizar um teste
de permutação entre os dois grupos.

Use:

- a **diferença entre médias** como estatística;
- amostras independentes;
- um teste bilateral;
- pelo menos 10.000 reamostragens.

Armazene o resultado do teste e examine:

- a estatística observada;
- o p-valor;
- a distribuição nula produzida pelo procedimento.

Consulte a documentação da função para identificar os argumentos necessários e
os atributos disponíveis no objeto retornado.


### 5.3. Visualizando a distribuição nula

Visualize a distribuição nula produzida pelo teste de permutação.

No mesmo gráfico, indique a posição da diferença observada entre os dois gêneros.
A documentação do Seaborn sobre [distribuições](https://seaborn.pydata.org/tutorial/distributions.html) pode servir como
referência para escolher uma representação adequada.

Observe onde o resultado original se encontra em relação aos valores que seriam
esperados sob a hipótese nula.


### 5.4. Interpretação do resultado

Responda:

1. Qual foi a diferença observada entre os grupos?
2. Qual foi o p-valor obtido?
3. Considerando $(\alpha = 0{,}05)$, qual é a decisão em relação a $(H_0)$? 

_(utilize f-strings para comunicar sua análise)_


## 6. Comparando os três gêneros

No teste de permutação, investigamos uma comparação específica entre dois grupos:
`mystery` e `romance`.

Nosso conjunto de dados contém ainda um terceiro gênero: `adventure`.

Agora ampliaremos a pergunta:

> **Há evidências de diferenças no comprimento médio das sentenças entre
> `adventure`, `mystery` e `romance`?**

Para investigar essa questão, utilizaremos uma **análise de variância de um fator
(ANOVA)**.


### 6.1. Verificando o comportamento dos três grupos antes da ANOVA

Na Seção 4.3, você já observou o comportamento geral de `mean_sentence_length`. Antes de ajustar o modelo de ANOVA, concentre-se nos pressupostos e na estrutura de variação dessa variável entre `adventure`, `mystery` e `romance`:

1. calcule a média, o desvio-padrão e o tamanho amostral ($n$) de `mean_sentence_length` para cada gênero;
2. produza um gráfico comparativo (como um `boxplot` combinado com `stripplot`, ou um `violinplot`) para inspecionar a dispersão e a simetria dentro de cada grupo;

### 6.2. ANOVA com statsmodels

Utilize o [statsmodels](https://www.statsmodels.org/stable/index.html) para testar
se o comprimento médio das sentenças difere entre os três gêneros.

A variável resposta será:

`mean_sentence_length`

e o fator será:

`genre`

Ajuste o modelo utilizando a fórmula:

`mean_sentence_length ~ C(genre)`

A documentação sobre [fórmulas no statsmodels](https://www.statsmodels.org/stable/examples/notebooks/generated/formulas.html) e
[`statsmodels.formula.api.ols()`](https://www.statsmodels.org/stable/generated/statsmodels.formula.api.ols.html) pode ser usada como
referência.

Em seguida, produza a tabela de ANOVA correspondente com
[`statsmodels.stats.anova.anova_lm()`](https://www.statsmodels.org/stable/generated/statsmodels.stats.anova.anova_lm.html).

Examine especialmente:

- os graus de liberdade;
- a soma dos quadrados;
- a estatística $F$;
- o valor-p.


### 6.3. Interpretando o teste global

A hipótese nula da ANOVA pode ser representada como:

$$
H_0:
\mu_{\text{adventure}}
=
\mu_{\text{mystery}}
=
\mu_{\text{romance}}
$$

A hipótese alternativa afirma que **nem todas as médias são iguais**.

A partir da tabela de ANOVA, responda:

1. Qual foi o valor da estatística $F$?
2. Qual foi o valor-p?
3. Considerando $(\alpha = 0{,}05)$, rejeitamos ou não rejeitamos $(H_0)$?
4. O que essa decisão significa no contexto dos três gêneros?


### 6.4. Limitações do teste global e conexão entre as análises

Se a ANOVA indicar evidência estatística contra $H_0$, concluímos que **pelo menos um dos gêneros possui média diferente dos demais**. O teste $F$ global (omnibus), entretanto, não informa *quais* pares específicos (`adventure` vs. `mystery`, `adventure` vs. `romance` ou `mystery` vs. `romance`) diferem entre si.

Para concluir o projeto, reflita:

1. Olhando para as médias e o gráfico da Seção 6.1, qual(is) gênero(s) parece(m) impulsionar o resultado encontrado na ANOVA?
2. Como o resultado da ANOVA na Seção 6 se conecta com o resultado do teste de permutação entre `mystery` e `romance` realizado na Seção 5?
3. Caso você quisesse testar formalmente todos os pares de gêneros após a ANOVA, por que não seria adequado simplesmente realizar vários testes $t$ ou testes de permutação independentes mantendo $\alpha = 0{,}05$ em cada um sem ajuste?

*(Opcional / Desafio)*: Utilize [`statsmodels.stats.multicomp.pairwise_tukeyhsd()`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) para realizar o teste *post-hoc* de Tukey e verifique quais pares de gêneros apresentam diferenças estatisticamente significativas.